In [ ]:
# ==============================================================================
# KAGGLE SETUP BLOCK (INDEPENDENT BLOCK)
# ==============================================================================
import os
import sys
import subprocess
from pathlib import Path

# Detect Kaggle environment
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if IS_KAGGLE:
    print("--- Detected Kaggle Environment ---")
    
    # 1. Clone repository if not present
    REPO_URL = "https://github.com/HCMUS-VIR-Nhom8/DINOv3-FAISS-HNSW-SOP-VisualProductSearch.git"
    REPO_DIR = Path("/kaggle/working/DINOv3-FAISS-HNSW-SOP-VisualProductSearch")
    
    if not REPO_DIR.exists():
        print(f"Cloning repository: {REPO_URL}")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

    # swich to branch "experiments"
    print("Switching to branch: experiments")
    try:
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "experiments"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error switching branch: {e}. Attempting to fetch first...")
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "experiments"], check=True)
    
    # Change working directory of the kernel to notebooks directory
    # so that PROJECT_ROOT = Path("..").resolve() works correctly
    NOTEBOOKS_DIR = REPO_DIR / "notebooks"
    if NOTEBOOKS_DIR.exists():
        os.chdir(str(NOTEBOOKS_DIR))
        print(f"Changed working directory to: {os.getcwd()}")
        
    # Add project root to sys.path so src imports work
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
        print("Added repository root to python path.")
        
    # 2. Download and unzip dataset from Google Drive
    DATA_DIR = REPO_DIR / "data" / "raw" / "Stanford_Online_Products"
    ZIP_PATH = REPO_DIR / "Stanford_Online_Products.zip"
    
    if not DATA_DIR.exists() or not (DATA_DIR / "Ebay_train.txt").exists():
        print("Dataset not found. Downloading from Google Drive...")
        
        # Install gdown if needed
        try:
            import gdown
        except ImportError:
            print("Installing gdown...")
            subprocess.run([sys.executable, "-m", "pip", "install", "gdown"], check=True)
            import gdown
            
        # Download Stanford Online Products ZIP
        file_id = "1TclrpQOF_ullUP99wk_gjGN8pKvtErG8"
        url = f"https://drive.google.com/uc?id={file_id}"
        print(f"Downloading from Google Drive ID: {file_id}")
        gdown.download(url, str(ZIP_PATH), quiet=False)
        
        # Extract the zip file
        print("Extracting dataset...")
        import zipfile
        raw_dir = REPO_DIR / "data" / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(str(ZIP_PATH), 'r') as zip_ref:
            zip_ref.extractall(str(raw_dir))
            
        print("Dataset extraction completed.")
        
        # Clean up zip file
        if ZIP_PATH.exists():
            ZIP_PATH.unlink()
            print("Cleaned up ZIP file.")
            
    # Note on HF login for gated DINOv3
    print("\n--- Hugging Face Access Note ---")
    print("DINOv3 is a gated Hugging Face model. If access token is required, run:")
    print("from huggingface_hub import login; login(token='YOUR_HF_TOKEN')\n")
else:
    print("Running in local environment. Setup skipped.")


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

# ---------------------------------------------------------------------------
# Dataset — PHẢI dùng đúng split đã tạo ở Notebook 02 (baseline). Không
# sampling lại, không tự tạo query/gallery split khác, để so sánh baseline
# vs proposed là fair (cùng gallery, cùng query, cùng ground-truth).
# ---------------------------------------------------------------------------
SPLIT_DIR = PROJECT_ROOT / "data" / "sampled"
SAMPLE_FILE = SPLIT_DIR / "sop_20k.csv"
GALLERY_FILE = SPLIT_DIR / "baseline_gallery.csv"
QUERY_FILE = SPLIT_DIR / "baseline_query.csv"

# Output proposed
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "proposed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUAL_DIR = OUTPUT_DIR / "qualitative"

EMBEDDING_FILE = OUTPUT_DIR / "gallery_embeddings.npy"
GALLERY_META_FILE = OUTPUT_DIR / "gallery_metadata.csv"
HNSW_INDEX_FILE = OUTPUT_DIR / "gallery_hnsw.index"
QUERY_RESULTS_FILE = OUTPUT_DIR / "retrieval_results.npy"
QUERY_SCORES_FILE = OUTPUT_DIR / "retrieval_scores.npy"
METRICS_FILE = OUTPUT_DIR / "metrics.json"
LATENCY_FILE = OUTPUT_DIR / "latency.json"
CONFIG_FILE = OUTPUT_DIR / "config.json"

# ---------------------------------------------------------------------------
# Model — DINOv3. Tên model lấy từ configs/proposed.yaml của repo, KHÔNG
# hard-code khác đi. DINOV3_LOCAL_PATH cho phép trỏ tới checkpoint cục bộ
# (thư mục chứa config.json/model weights kiểu HuggingFace) thay vì Hub.
# ---------------------------------------------------------------------------
DINOV3_MODEL_NAME = "facebook/dinov3-vitb16-pretrain-lvd1689m"  # configs/proposed.yaml -> model.name
DINOV3_LOCAL_PATH = None  # vd: "/path/to/local/dinov3-checkpoint"
MODEL_SOURCE = DINOV3_LOCAL_PATH if DINOV3_LOCAL_PATH else DINOV3_MODEL_NAME

# ---------------------------------------------------------------------------
# Preprocessing — theo đúng ProposedPreprocessor (src/preprocessing/pipeline.py)
# và giá trị mặc định trong configs/proposed.yaml. Localization/segmentation
# (Grounding DINO / SAM) KHÔNG được bật: tài liệu phương pháp (mục 4.1.2.3)
# mô tả đây là các thành phần "có thể thay thế" trong prototype, và repo
# hiện chưa gắn checkpoint thật cho hai bước này (CONFIGURABLE, không phải
# method chính thức bắt buộc).
# ---------------------------------------------------------------------------
RESIZE_LONG_SIDE = 1024
TARGET_SIZE = 518            # kích thước sau letterbox
IMAGE_SIZE = TARGET_SIZE     # alias cho đúng tên biến yêu cầu trong đề bài
PADDING_RATIO = 0.10          # chưa dùng vì localization/segmentation đang tắt
BLUR_THRESHOLD = 50.0
JPEG_THRESHOLD = 8.0
USE_ILLUMINATION_CORRECTION = True
CLAHE_CLIP_LIMIT = 2.0

BATCH_SIZE = 16
NUM_WORKERS = 2

# ---------------------------------------------------------------------------
# FAISS HNSW — mặc định lấy từ configs/proposed.yaml (mục 4.1.7 trong tài
# liệu phương pháp). Đổi ở đây để thử nghiệm đánh đổi recall/latency/memory.
# ---------------------------------------------------------------------------
HNSW_M = 32
HNSW_EF_CONSTRUCTION = 200
HNSW_EF_SEARCH = 64

# ---------------------------------------------------------------------------
# Metadata re-ranking — dùng ĐÚNG công thức đã hiện thực trong
# src/reranking/metadata.py::category_consistency_rerank (không tự bịa công
# thức khác — xem mục 14 của yêu cầu).
#
# LƯU Ý QUAN TRỌNG (xem README của repo): SOP không có metadata dạng
# title/brand/mô tả văn bản cho ảnh truy vấn. Do đó công thức này KHÔNG so
# khớp metadata của query với candidate (không phải oracle match), mà tính
# điểm đồng thuận (consensus) super_class_id ngay trong tập Top-N candidate:
#
#     meta_score(candidate) = tần suất super_class_id của candidate đó
#                              trong Top-N candidates / N
#     final_score = alpha_visual * visual_score_normalized
#                   + (1 - alpha_visual) * meta_score
#
# Đây là một CONFIGURABLE EXPERIMENTAL CHOICE đã được README của repo xác
# nhận, KHÔNG PHẢI một công thức chính thức đã được đề tài chứng minh tối
# ưu. Nếu có metadata text/brand thật, hãy thay hàm rerank tương ứng thay vì
# coi category-consistency là mặc định vĩnh viễn.
# ---------------------------------------------------------------------------
ENABLE_RERANKING = True
RERANK_CANDIDATES = 100      # = retrieval.candidate_k trong configs/proposed.yaml
ALPHA_VISUAL = 0.85          # = reranking.alpha_visual trong configs/proposed.yaml
VISUAL_WEIGHT = ALPHA_VISUAL
METADATA_WEIGHT = 1.0 - ALPHA_VISUAL

# ---------------------------------------------------------------------------
# Evaluation — PHẢI trùng KS đã dùng ở Notebook 02 để so sánh công bằng.
# ---------------------------------------------------------------------------
KS = [1, 5, 10, 20, 50, 100]
MAX_QUERIES = None           # None = toàn bộ query set
RANDOM_SEED = 42

# Device — đổi thành "cpu" nếu không có GPU.
DEVICE = "cuda"

N_QUALITATIVE = 20            # số lượng ảnh minh họa xuất ra qualitative/

print("PROJECT_ROOT :", PROJECT_ROOT)
print("GALLERY_FILE :", GALLERY_FILE)
print("QUERY_FILE   :", QUERY_FILE)
print("OUTPUT_DIR   :", OUTPUT_DIR)
print("MODEL_SOURCE :", MODEL_SOURCE)

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import torch
import transformers

try:
    import faiss
except ImportError as e:
    raise ImportError(
        "Không import được faiss. Cài đặt bằng:\n"
        "  pip install faiss-cpu   # hoặc faiss-gpu nếu có CUDA phù hợp\n"
        f"Lỗi gốc: {e}"
    )

# Cho phép "from src...." hoạt động khi notebook chạy từ thư mục notebooks/
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("Transformers:", transformers.__version__)
print("FAISS       :", getattr(faiss, "__version__", "n/a"))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
elif DEVICE == "cuda":
    print(
        "\u26a0 DEVICE='cuda' nhưng không có GPU khả dụng. "
        "Hãy đổi DEVICE='cpu' ở Cell Configuration trước khi chạy tiếp."
    )

In [ ]:
# Theo mục 23 của yêu cầu: dùng lại module đã có trong src/ thay vì viết lại.
try:
    from src.preprocessing.pipeline import ProposedPreprocessor
    from src.models.encoder import DINOv3Encoder
    from src.retrieval.hnsw import HNSWRetriever
    from src.reranking.metadata import category_consistency_rerank
except ImportError as e:
    raise ImportError(
        "Không import được các module trong src/. Kiểm tra:\n"
        "  1. Notebook đang chạy từ thư mục notebooks/ (PROJECT_ROOT = ..)\n"
        "  2. Repo có đầy đủ src/preprocessing/pipeline.py, src/models/encoder.py, "
        "src/retrieval/hnsw.py, src/reranking/metadata.py\n"
        f"Lỗi gốc: {e}"
    )

print("\u2713 Đã import ProposedPreprocessor, DINOv3Encoder, HNSWRetriever, "
      "category_consistency_rerank từ src/.")

In [ ]:
gallery_df = pd.read_csv(GALLERY_FILE).reset_index(drop=True)
query_df = pd.read_csv(QUERY_FILE).reset_index(drop=True)

print("Gallery:", len(gallery_df), "| classes:", gallery_df["class_id"].nunique())
print("Query  :", len(query_df), "| classes:", query_df["class_id"].nunique())

display(gallery_df.head())

In [ ]:
required_columns = {"image_id", "class_id", "image_path"}

for name, df in [("gallery_df", gallery_df), ("query_df", query_df)]:
    missing_cols = required_columns - set(df.columns)
    if missing_cols:
        raise ValueError(f"{name} thiếu cột bắt buộc: {missing_cols}")

# Mỗi query class phải có ít nhất 1 ảnh trong gallery (positive tồn tại)
missing_query_classes = set(query_df["class_id"]) - set(gallery_df["class_id"])
assert len(missing_query_classes) == 0, (
    f"{len(missing_query_classes)} class trong query không có ảnh gallery tương ứng."
)

# Query và gallery không được trùng image_id (tránh self-retrieval ảo)
overlap = set(query_df["image_id"]) & set(gallery_df["image_id"])
assert len(overlap) == 0, f"Query và gallery bị overlap {len(overlap)} image_id."

# Kiểm tra image path tồn tại trên đĩa
for name, df in [("gallery_df", gallery_df), ("query_df", query_df)]:
    exists = df["image_path"].map(os.path.exists)
    if not exists.all():
        n_missing = int((~exists).sum())
        raise FileNotFoundError(f"{name} có {n_missing} image_path không tồn tại trên đĩa.")

# Reranking cần super_class_id — nếu thiếu, báo lỗi rõ ràng thay vì âm thầm tắt.
if ENABLE_RERANKING and "super_class_id" not in gallery_df.columns:
    raise ValueError(
        "ENABLE_RERANKING=True nhưng gallery_df không có cột 'super_class_id'.\n"
        "category_consistency_rerank() cần cột này để tính metadata score.\n"
        "Hãy đảm bảo Notebook 01 giữ lại cột super_class_id khi tạo sop_20k.csv, "
        "hoặc đặt ENABLE_RERANKING = False ở Cell Configuration để chạy HNSW-only."
    )

print(
    "\u2713 Split hợp lệ: không overlap, mọi query class đều có gallery, "
    "mọi image_path tồn tại"
    + (", có super_class_id cho reranking." if ENABLE_RERANKING else ".")
)

In [ ]:
try:
    dinov3 = DINOv3Encoder(MODEL_SOURCE, DEVICE)
except Exception as e:
    raise RuntimeError(
        f"Không load được DINOv3 ({MODEL_SOURCE}).\n"
        "Nguyên nhân thường gặp:\n"
        "  1. Chưa đăng nhập Hugging Face hoặc chưa được cấp quyền truy cập "
        "model gated -> chạy `huggingface-cli login` và xin quyền tại trang "
        "model card trên huggingface.co.\n"
        "  2. Không có kết nối mạng tới huggingface.co.\n"
        "  3. DINOV3_LOCAL_PATH (Cell Configuration) trỏ sai đường dẫn checkpoint cục bộ.\n"
        f"Lỗi gốc: {e}"
    )

print("\u2713 Đã load DINOv3Encoder")
print("Device       :", dinov3.device)
print("Embedding dim:", dinov3.dim)

In [ ]:
# LƯU Ý: DINOv3Encoder.__init__ (src/models/encoder.py) hiện chỉ gọi
# .to(device).eval() và CHƯA tự set requires_grad_(False) cho tham số.
# Để đúng yêu cầu "frozen backbone" của tài liệu phương pháp (mục 4.1.3.3:
# "Có thể sử dụng như một backbone đóng băng"), notebook tự đóng băng tường
# minh ở đây.
# TODO (repo): cân nhắc chuyển đoạn freeze này vào DINOv3Encoder.__init__.
for p in dinov3.model.parameters():
    p.requires_grad_(False)

num_params = sum(p.numel() for p in dinov3.model.parameters())
num_trainable = sum(p.numel() for p in dinov3.model.parameters() if p.requires_grad)

print("Tổng số tham số     :", f"{num_params:,}")
print("Tham số có thể train:", f"{num_trainable:,}")
print("model.training =", dinov3.model.training)

assert num_trainable == 0, "DINOv3 backbone chưa được đóng băng hoàn toàn."
assert dinov3.model.training is False, "DINOv3 model chưa ở chế độ eval()."

print("\u2713 Backbone đã đóng băng và ở chế độ eval — đúng yêu cầu frozen backbone.")

In [ ]:
# Load offline proposed embeddings and HNSW index
from src.retrieval.hnsw import HNSWRetriever

if not EMBEDDING_FILE.exists() or not GALLERY_META_FILE.exists() or not HNSW_INDEX_FILE.exists():
    raise FileNotFoundError(
        f"Offline proposed embeddings/index not found under {OUTPUT_DIR}. "
        f"Please run the offline notebook (03_proposed_offline.ipynb) first."
    )

gallery_embeddings = np.load(EMBEDDING_FILE)
gallery_df = pd.read_csv(GALLERY_META_FILE).reset_index(drop=True)
hnsw_retriever = HNSWRetriever.load(HNSW_INDEX_FILE)

# Placeholders for variables referenced in evaluation
hnsw_build_time = 0.0
gallery_encode_time = 0.0

print("Loaded offline proposed artifacts:")
print("Gallery Embeddings Shape:", gallery_embeddings.shape)
print("HNSW Index ntotal       :", hnsw_retriever.index.ntotal)


In [ ]:
proposed_preprocessor = ProposedPreprocessor(
    resize_long_side=RESIZE_LONG_SIDE,
    target_size=TARGET_SIZE,
    padding_ratio=PADDING_RATIO,
    blur_threshold=BLUR_THRESHOLD,
    jpeg_threshold=JPEG_THRESHOLD,
    use_illumination=USE_ILLUMINATION_CORRECTION,
    clahe_clip_limit=CLAHE_CLIP_LIMIT,
)

preview_row = gallery_df.iloc[0]
preview_original = Image.open(preview_row["image_path"]).convert("RGB")
preview_output = proposed_preprocessor(preview_row["image_path"])

plt.figure(figsize=(9, 4))
ax = plt.subplot(1, 2, 1)
ax.imshow(preview_original)
ax.set_title("Original")
ax.axis("off")

ax = plt.subplot(1, 2, 2)
ax.imshow(preview_output.image)
ax.set_title(f"Proposed preprocessing: {TARGET_SIZE}\u00d7{TARGET_SIZE}")
ax.axis("off")
plt.tight_layout()
plt.show()

print("blur_score:", preview_output.blur_score, "| flags:", preview_output.metadata)
print(
    "Lưu ý: AutoImageProcessor của HuggingFace sẽ áp dụng thêm resize/normalize "
    "riêng của DINOv3 lên ảnh đã letterbox này khi gọi dinov3.encode() — đây là "
    "thiết kế sẵn có của repo (DINOv3Encoder), không phải augmentation bổ sung "
    "do notebook này thêm vào."
)

In [ ]:
def encode_query(image_path, preprocessor, encoder, device):
    """
    Online query encoding: load ảnh -> preprocessing đề xuất -> DINOv3 -> L2.

    Trả về (embedding [1, D], elapsed_seconds). Đo latency RIÊNG cho bước
    này — không gộp với search/rerank (mục 11 và 17 của yêu cầu).
    """
    out = preprocessor(image_path)

    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()

    embedding = encoder.encode([out.image])  # đã L2-normalize bên trong

    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    return embedding, elapsed

### Cell 22 — Test encode_query trên 1 query

In [ ]:
query_row = query_df.iloc[0]
q_embedding, q_encode_time = encode_query(
    query_row["image_path"], proposed_preprocessor, dinov3, DEVICE
)

print("Query image_id:", query_row["image_id"], "| GT class:", query_row["class_id"])
print("Embedding shape:", q_embedding.shape)
print(f"Encoding latency: {q_encode_time * 1000:.2f} ms")
assert np.allclose(np.linalg.norm(q_embedding, axis=1), 1.0, atol=1e-4)

## 9. HNSW ANN Search

### Cell 23 — Hàm hnsw_search()

In [ ]:
def hnsw_search(query_embedding, index, candidate_k):
    """
    ANN search trên FAISS HNSW. Trả về (ids, scores, elapsed_seconds).
    Đo latency RIÊNG cho search, tách khỏi encoding và reranking.
    """
    start = time.perf_counter()
    scores, ids = index.search(query_embedding, candidate_k)
    elapsed = time.perf_counter() - start
    return ids, scores, elapsed


q_ids, q_scores, q_search_time = hnsw_search(q_embedding, hnsw_retriever, RERANK_CANDIDATES)
print(f"Search latency: {q_search_time * 1000:.2f} ms")
print("Top-5 candidate ids    :", q_ids[0, :5])
print("Top-5 candidate scores :", q_scores[0, :5])

### Cell 24 — Xem Top-10 kết quả (trước reranking)

In [ ]:
def safe_class_lookup(ids_row, class_id_array):
    """
    Tra class_id theo vị trí, KHÔNG dùng numpy negative-index cho id=-1
    (FAISS trả -1 khi không đủ candidate) — tránh lỗi âm thầm trỏ nhầm
    phần tử cuối của mảng (mục 28 của yêu cầu: không silently ignore lỗi).
    """
    ids_row = np.asarray(ids_row)
    out = np.full(ids_row.shape, -1, dtype="int64")
    valid = ids_row >= 0
    out[valid] = np.asarray(class_id_array)[ids_row[valid]]
    return out


rows = []
retrieved_classes_preview = safe_class_lookup(q_ids[0][:10], gallery_df["class_id"].to_numpy())
for rank, (idx, score, cls) in enumerate(
    zip(q_ids[0][:10], q_scores[0][:10], retrieved_classes_preview), start=1
):
    rows.append({
        "rank": rank,
        "gallery_index": int(idx),
        "class_id": int(cls),
        "similarity": float(score),
        "correct": int(cls) == int(query_row["class_id"]),
    })

display(pd.DataFrame(rows))

## 10. Metadata Re-ranking

Dùng đúng hàm `category_consistency_rerank` trong
`src/reranking/metadata.py` (công thức đã nêu ở Cell Configuration). Đây
KHÔNG phải re-ranking theo metadata của chính ảnh truy vấn (vì SOP không có
metadata dạng đó cho query), mà là đồng thuận `super_class_id` bên trong tập
Top-N candidate — đúng những gì repo hiện có, không tự bịa công thức khác
(mục 14 của yêu cầu).

### Cell 25 — Hàm rerank_candidates() + test trên 1 query

In [ ]:
def rerank_candidates(scores, ids, gallery_df, alpha_visual):
    """Wrapper mỏng quanh category_consistency_rerank(), đo latency riêng."""
    start = time.perf_counter()
    reranked_scores, reranked_ids = category_consistency_rerank(
        scores, ids, gallery_df, alpha_visual=alpha_visual
    )
    elapsed = time.perf_counter() - start
    return reranked_ids, reranked_scores, elapsed


if ENABLE_RERANKING:
    q_ids_rerank, q_scores_rerank, q_rerank_time = rerank_candidates(
        q_scores, q_ids, gallery_df, ALPHA_VISUAL
    )
    print(f"Rerank latency: {q_rerank_time * 1000:.2f} ms")

    rows = []
    retrieved_classes_rerank = safe_class_lookup(
        q_ids_rerank[0][:10], gallery_df["class_id"].to_numpy()
    )
    for rank, (idx, score, cls) in enumerate(
        zip(q_ids_rerank[0][:10], q_scores_rerank[0][:10], retrieved_classes_rerank), start=1
    ):
        rows.append({
            "rank": rank,
            "gallery_index": int(idx),
            "class_id": int(cls),
            "final_score": float(score),
            "correct": int(cls) == int(query_row["class_id"]),
        })
    print("Top-10 SAU reranking:")
    display(pd.DataFrame(rows))
else:
    print("ENABLE_RERANKING = False -> bỏ qua bước reranking (dùng thẳng kết quả HNSW).")

## 11. End-to-End Retrieval

### Cell 26 — Hàm proposed_retrieve() (end-to-end)

In [ ]:
def proposed_retrieve(image_path, preprocessor, encoder, index, gallery_df,
                       candidate_k, final_k, enable_reranking, alpha_visual, device):
    """
    Pipeline online đầy đủ: encode -> HNSW search -> (tuỳ chọn) rerank -> Top-K.

    Trả về dict:
        indices, scores, encoding_time, search_time, rerank_time, end_to_end_time
    """
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    embedding, encoding_time = encode_query(image_path, preprocessor, encoder, device)
    ids, scores, search_time = hnsw_search(embedding, index, candidate_k)

    if enable_reranking:
        ids, scores, rerank_time = rerank_candidates(scores, ids, gallery_df, alpha_visual)
    else:
        rerank_time = 0.0

    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
    end_to_end_time = time.perf_counter() - t_start

    return {
        "indices": ids[:, :final_k],
        "scores": scores[:, :final_k],
        "encoding_time": encoding_time,
        "search_time": search_time,
        "rerank_time": rerank_time,
        "end_to_end_time": end_to_end_time,
    }

### Cell 27 — Test proposed_retrieve() trên 1 query

In [ ]:
result = proposed_retrieve(
    query_row["image_path"], proposed_preprocessor, dinov3, hnsw_retriever, gallery_df,
    candidate_k=RERANK_CANDIDATES, final_k=max(KS),
    enable_reranking=ENABLE_RERANKING, alpha_visual=ALPHA_VISUAL, device=DEVICE,
)

print(f"Encoding  : {result['encoding_time'] * 1000:.2f} ms")
print(f"Search    : {result['search_time'] * 1000:.2f} ms")
print(f"Rerank    : {result['rerank_time'] * 1000:.2f} ms")
print(f"End-to-end: {result['end_to_end_time'] * 1000:.2f} ms")

# EVALUATION

## 12. Recall@K & 13. mAP

### Cell 28 — Offline: extract query embeddings cho toàn bộ eval set

In [ ]:
if MAX_QUERIES is None:
    eval_query_df = query_df.copy()
else:
    eval_query_df = query_df.head(MAX_QUERIES).copy()

if DEVICE == "cuda" and torch.cuda.is_available():
    torch.cuda.synchronize()

query_embeddings, query_image_ids, query_class_ids, query_quality, query_encode_time = \
    extract_dinov3_embeddings(
        eval_query_df, proposed_preprocessor, dinov3, BATCH_SIZE, desc="Query embeddings"
    )

if DEVICE == "cuda" and torch.cuda.is_available():
    torch.cuda.synchronize()

print("Query embedding shape:", query_embeddings.shape)
print("Query extraction time:", query_encode_time, "sec")

### Cell 29 — HNSW search cho toàn bộ query

In [ ]:
if DEVICE == "cuda" and torch.cuda.is_available():
    torch.cuda.synchronize()

search_start = time.perf_counter()
retrieval_ids_pre_rerank, retrieval_scores_pre_rerank = hnsw_retriever.search(
    query_embeddings, RERANK_CANDIDATES
)
total_search_time = time.perf_counter() - search_start

if DEVICE == "cuda" and torch.cuda.is_available():
    torch.cuda.synchronize()

num_eval_queries = len(eval_query_df)
avg_search_ms = total_search_time / num_eval_queries * 1000

print("Queries:", num_eval_queries)
print("Total HNSW search:", total_search_time, "sec")
print("Average search/query:", avg_search_ms, "ms")

### Cell 30 — Metadata reranking cho toàn bộ query (nếu bật)

In [ ]:
if ENABLE_RERANKING:
    rerank_start = time.perf_counter()
    retrieval_ids, retrieval_scores = category_consistency_rerank(
        retrieval_scores_pre_rerank, retrieval_ids_pre_rerank, gallery_df,
        alpha_visual=ALPHA_VISUAL,
    )
    total_rerank_time = time.perf_counter() - rerank_start
else:
    retrieval_ids, retrieval_scores = retrieval_ids_pre_rerank, retrieval_scores_pre_rerank
    total_rerank_time = 0.0

print("Total rerank time:", total_rerank_time, "sec")

### Cell 31 — Hàm Recall@K và Average Precision (giống hệt Notebook 02)

In [ ]:
def recall_at_k(retrieved_class_ids, ground_truth_class_id, k):
    retrieved = np.asarray(retrieved_class_ids[:k])
    retrieved = retrieved[retrieved != -1]
    return float(np.any(retrieved == ground_truth_class_id))


def average_precision(retrieved_class_ids, ground_truth_class_id):
    """
    AP cho instance/product retrieval — mọi ảnh gallery cùng class_id được
    coi là relevant. Giống hệt công thức đã dùng ở Notebook 02 để bảo đảm
    so sánh công bằng (không dùng super_class_id làm ground truth).
    """
    retrieved = np.asarray(retrieved_class_ids)
    retrieved = retrieved[retrieved != -1]
    relevant = (retrieved == ground_truth_class_id)
    num_relevant = relevant.sum()
    if num_relevant == 0:
        return 0.0

    precisions = []
    hits = 0
    for rank, is_relevant in enumerate(relevant, start=1):
        if is_relevant:
            hits += 1
            precisions.append(hits / rank)
    return float(np.mean(precisions))


def class_matrix_from_ids(ids_matrix, class_id_array):
    """Tra class_id cho cả ma trận [N, K] ids, an toàn với id=-1 (padding)."""
    flat = ids_matrix.reshape(-1)
    out = np.full(flat.shape, -1, dtype="int64")
    valid = flat >= 0
    out[valid] = np.asarray(class_id_array)[flat[valid]]
    return out.reshape(ids_matrix.shape)

### Cell 32 — Tính Recall@K và mAP

In [ ]:
retrieved_class_matrix = class_matrix_from_ids(retrieval_ids, gallery_df["class_id"].to_numpy())

metrics = {}
for k in KS:
    recalls = [
        recall_at_k(retrieved_class_matrix[i], query_class_ids[i], k)
        for i in range(num_eval_queries)
    ]
    metrics[f"Recall@{k}"] = float(np.mean(recalls))

aps = [
    average_precision(retrieved_class_matrix[i], query_class_ids[i])
    for i in range(num_eval_queries)
]
metrics["mAP"] = float(np.mean(aps))

print(json.dumps(metrics, indent=2))

### Cell 33 — Kiểm tra Recall@K có tính đơn điệu

In [ ]:
recall_values = [metrics[f"Recall@{k}"] for k in KS]
print(list(zip(KS, recall_values)))

for a, b in zip(recall_values, recall_values[1:]):
    assert b + 1e-12 >= a

print("\u2713 Recall@K không giảm khi K tăng.")

## 14. Latency

Đo riêng 4 thành phần latency (đơn vị ms), giống protocol của Notebook
02 nhưng có thêm bước `rerank`:

```text
1. DINOv3 query encoding
2. HNSW search
3. metadata re-ranking
4. end-to-end
```

Nếu CUDA khả dụng, `torch.cuda.synchronize()` được gọi trước/sau mỗi đoạn đo
để tránh đo sai do thực thi bất đồng bộ trên GPU (đã cài trong
`encode_query`/`proposed_retrieve`).

### Cell 34 — Phân tích latency per-query (encoding/search/rerank/end-to-end)

In [ ]:
encoding_times, search_times, rerank_times, end_to_end_times = [], [], [], []

for _, row in tqdm(eval_query_df.iterrows(), total=len(eval_query_df), desc="Latency benchmark"):
    result = proposed_retrieve(
        row["image_path"], proposed_preprocessor, dinov3, hnsw_retriever, gallery_df,
        candidate_k=RERANK_CANDIDATES, final_k=max(KS),
        enable_reranking=ENABLE_RERANKING, alpha_visual=ALPHA_VISUAL, device=DEVICE,
    )
    encoding_times.append(result["encoding_time"])
    search_times.append(result["search_time"])
    rerank_times.append(result["rerank_time"])
    end_to_end_times.append(result["end_to_end_time"])


def _ms_stats(values):
    values = np.asarray(values) * 1000
    return {
        "mean_ms": float(np.mean(values)),
        "p50_ms": float(np.percentile(values, 50)),
        "p95_ms": float(np.percentile(values, 95)),
    }


enc_stats = _ms_stats(encoding_times)
search_stats = _ms_stats(search_times)
rerank_stats = _ms_stats(rerank_times)
e2e_stats = _ms_stats(end_to_end_times)

latency_stats = {
    "encoding_mean_ms": enc_stats["mean_ms"],
    "encoding_p50_ms": enc_stats["p50_ms"],
    "encoding_p95_ms": enc_stats["p95_ms"],
    "search_mean_ms": search_stats["mean_ms"],
    "search_p50_ms": search_stats["p50_ms"],
    "search_p95_ms": search_stats["p95_ms"],
    "rerank_mean_ms": rerank_stats["mean_ms"],
    "rerank_p50_ms": rerank_stats["p50_ms"],
    "rerank_p95_ms": rerank_stats["p95_ms"],
    "end_to_end_mean_ms": e2e_stats["mean_ms"],
    "end_to_end_p50_ms": e2e_stats["p50_ms"],
    "end_to_end_p95_ms": e2e_stats["p95_ms"],
}

print(json.dumps(latency_stats, indent=2))

## 15. Memory

### Cell 35 — Đo memory (embedding, HNSW index, GPU model memory riêng biệt)

In [ ]:
gallery_embedding_mb = gallery_embeddings.nbytes / (1024 ** 2)
query_embedding_mb = query_embeddings.nbytes / (1024 ** 2)
hnsw_index_mb = HNSW_INDEX_FILE.stat().st_size / (1024 ** 2)
candidate_scores_mb = retrieval_scores_pre_rerank.nbytes / (1024 ** 2)

memory_stats = {
    "gallery_embeddings_MB": float(gallery_embedding_mb),
    "query_embeddings_MB": float(query_embedding_mb),
    "hnsw_index_MB": float(hnsw_index_mb),
    "candidate_scores_MB": float(candidate_scores_mb),
    "feature_dimension": int(dinov3.dim),
    "gallery_size": int(len(gallery_df)),
}

# GPU memory (nếu có) được báo cáo RIÊNG — đây là bộ nhớ model DINOv3 trên
# GPU, KHÔNG được gộp vào "retrieval index memory" (HNSW index nằm trên
# CPU/disk) — theo đúng yêu cầu mục 18.
if DEVICE == "cuda" and torch.cuda.is_available():
    memory_stats["gpu_model_memory_MB"] = float(
        torch.cuda.max_memory_allocated() / (1024 ** 2)
    )
else:
    memory_stats["gpu_model_memory_MB"] = None

print(json.dumps(memory_stats, indent=2))

### Cell 36 — Lưu retrieval results

In [ ]:
np.save(QUERY_RESULTS_FILE, retrieval_ids)
np.save(QUERY_SCORES_FILE, retrieval_scores)

print("Saved:", QUERY_RESULTS_FILE)
print("Saved:", QUERY_SCORES_FILE)

### Cell 37 — Lưu metrics.json, latency.json, config.json

In [ ]:
final_metrics = {
    **metrics,
    "num_gallery": int(len(gallery_df)),
    "num_queries": int(num_eval_queries),
    "feature_dim": int(dinov3.dim),
    "gallery_embedding_MB": float(gallery_embedding_mb),
    "gallery_encoding_time_sec": float(gallery_encode_time),
    "query_encoding_time_sec": float(query_encode_time),
    "hnsw_build_time_sec": float(hnsw_build_time),
    "hnsw_search_total_sec": float(total_search_time),
    "hnsw_search_avg_ms": float(avg_search_ms),
    "reranking_enabled": bool(ENABLE_RERANKING),
    "rerank_total_sec": float(total_rerank_time),
    **latency_stats,
    **memory_stats,
}

with open(METRICS_FILE, "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, indent=2)

with open(LATENCY_FILE, "w", encoding="utf-8") as f:
    json.dump(latency_stats, f, indent=2)

run_config = {
    "model": {
        "name": DINOV3_MODEL_NAME, "local_path": DINOV3_LOCAL_PATH, "source": MODEL_SOURCE,
        "feature_dim": int(dinov3.dim), "device": DEVICE,
    },
    "preprocessing": {
        "resize_long_side": RESIZE_LONG_SIDE, "target_size": TARGET_SIZE,
        "padding_ratio": PADDING_RATIO, "blur_threshold": BLUR_THRESHOLD,
        "jpeg_threshold": JPEG_THRESHOLD,
        "illumination_correction": USE_ILLUMINATION_CORRECTION,
        "clahe_clip_limit": CLAHE_CLIP_LIMIT,
        "localization": "disabled", "segmentation": "disabled",
    },
    "retrieval": {
        "type": "hnsw", "M": HNSW_M, "efConstruction": HNSW_EF_CONSTRUCTION,
        "efSearch": HNSW_EF_SEARCH, "candidate_k": RERANK_CANDIDATES,
    },
    "reranking": {
        "enabled": ENABLE_RERANKING, "mode": "category_consistency",
        "alpha_visual": ALPHA_VISUAL, "top_n": RERANK_CANDIDATES,
    },
    "evaluation": {"ks": KS, "num_queries": int(num_eval_queries), "seed": RANDOM_SEED},
}

with open(CONFIG_FILE, "w", encoding="utf-8") as f:
    json.dump(run_config, f, indent=2)

print("Saved:", METRICS_FILE)
print("Saved:", LATENCY_FILE)
print("Saved:", CONFIG_FILE)
print(json.dumps(final_metrics, indent=2))

### Cell 38 — Plot Recall@K

In [ ]:
plt.figure(figsize=(7, 5))
label = "Proposed (DINOv3+HNSW" + ("+rerank)" if ENABLE_RERANKING else ")")
plt.plot(KS, [metrics[f"Recall@{k}"] for k in KS], marker="o", label=label)
plt.xlabel("K")
plt.ylabel("Recall@K")
plt.title("Proposed Recall@K")
plt.xticks(KS)
plt.grid(alpha=0.2)
plt.legend()

recall_plot = OUTPUT_DIR / "recall_at_k.png"
plt.savefig(recall_plot, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", recall_plot)

## 16. HNSW vs HNSW + Re-ranking (ablation)

So sánh **A: DINOv3 + HNSW** với **B: DINOv3 + HNSW + metadata
re-ranking**, dùng CHUNG tập candidate Top-N đã lấy từ HNSW (không
encode/search lại) — nên phần chênh lệch Recall@K/mAP đến từ đúng một biến
duy nhất: có hay không bước metadata re-ranking.

### Cell 39 — Tính Recall@K/mAP cho HNSW-only vs HNSW+reranking

In [ ]:
retrieved_class_matrix_hnsw_only = class_matrix_from_ids(
    retrieval_ids_pre_rerank, gallery_df["class_id"].to_numpy()
)

metrics_hnsw_only = {}
for k in KS:
    recalls = [
        recall_at_k(retrieved_class_matrix_hnsw_only[i], query_class_ids[i], k)
        for i in range(num_eval_queries)
    ]
    metrics_hnsw_only[f"Recall@{k}"] = float(np.mean(recalls))

aps_hnsw_only = [
    average_precision(retrieved_class_matrix_hnsw_only[i], query_class_ids[i])
    for i in range(num_eval_queries)
]
metrics_hnsw_only["mAP"] = float(np.mean(aps_hnsw_only))

ablation_df = pd.DataFrame({
    "HNSW only": metrics_hnsw_only,
    "HNSW + reranking": metrics,
})
display(ablation_df)

ablation_df.to_csv(OUTPUT_DIR / "ablation_hnsw_vs_reranking.csv")
print("Saved:", OUTPUT_DIR / "ablation_hnsw_vs_reranking.csv")

if not ENABLE_RERANKING:
    print(
        "\u26a0 ENABLE_RERANKING=False ở Cell Configuration -> hai cột trên "
        "giống hệt nhau (đều là HNSW-only). Bật ENABLE_RERANKING=True và chạy "
        "lại notebook để xem tác động thật của reranking."
    )
else:
    for k in KS:
        delta = metrics[f"Recall@{k}"] - metrics_hnsw_only[f"Recall@{k}"]
        sign = "cải thiện" if delta > 0 else ("không đổi" if delta == 0 else "giảm")
        print(f"Recall@{k}: reranking {sign} {abs(delta):.4f}")
    delta_map = metrics["mAP"] - metrics_hnsw_only["mAP"]
    print(f"mAP: reranking {'cải thiện' if delta_map > 0 else 'giảm'} {abs(delta_map):.4f}")

# QUALITATIVE

## 17. Retrieval Visualization

### Cell 40 — Chọn 1 query để minh họa

In [ ]:
QUAL_QUERY_INDEX = 0
QUAL_TOP_K = 5

q = eval_query_df.iloc[QUAL_QUERY_INDEX]
result = proposed_retrieve(
    q["image_path"], proposed_preprocessor, dinov3, hnsw_retriever, gallery_df,
    candidate_k=RERANK_CANDIDATES, final_k=QUAL_TOP_K,
    enable_reranking=ENABLE_RERANKING, alpha_visual=ALPHA_VISUAL, device=DEVICE,
)

q_indices = result["indices"][0]
q_scores_top = result["scores"][0]

print("Query image:", q["image_path"])
print("GT class:", q["class_id"])

qual_rows = []
for rank, (idx, score) in enumerate(zip(q_indices, q_scores_top), start=1):
    if idx < 0:
        continue
    g = gallery_df.iloc[int(idx)]
    qual_rows.append({
        "rank": rank,
        "image_id": int(g["image_id"]),
        "class_id": int(g["class_id"]),
        "similarity": float(score),
        "correct": int(g["class_id"]) == int(q["class_id"]),
        "image_path": g["image_path"],
    })

qual_df = pd.DataFrame(qual_rows)
display(qual_df)

### Cell 41 — Vẽ query + Top-K

In [ ]:
def show_proposed_retrieval(query_row, result_df, save_path=None):
    n = len(result_df)
    plt.figure(figsize=(3 * (n + 1), 4))

    ax = plt.subplot(1, n + 1, 1)
    ax.imshow(Image.open(query_row["image_path"]).convert("RGB"))
    ax.set_title(f"QUERY\nclass={query_row['class_id']}")
    ax.axis("off")

    for i, (_, row) in enumerate(result_df.iterrows(), start=1):
        ax = plt.subplot(1, n + 1, i + 1)
        ax.imshow(Image.open(row["image_path"]).convert("RGB"))
        correct_text = "\u2713" if row["correct"] else "\u2717"
        ax.set_title(
            f"Top-{row['rank']} {correct_text}\n"
            f"class={row['class_id']}\nscore={row['similarity']:.3f}"
        )
        ax.axis("off")

    title = "Proposed: DINOv3 + HNSW" + (" + Metadata Re-ranking" if ENABLE_RERANKING else "")
    plt.suptitle(title)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()


qual_path = OUTPUT_DIR / "qualitative_query_0000.png"
show_proposed_retrieval(q, qual_df, save_path=qual_path)
print("Saved:", qual_path)

## 18. Export Qualitative Results

### Cell 42 — Xuất nhiều qualitative cases

In [ ]:
QUAL_DIR.mkdir(parents=True, exist_ok=True)
n_qual = min(N_QUALITATIVE, len(eval_query_df))

for qi in tqdm(range(n_qual), desc="Export qualitative examples"):
    q = eval_query_df.iloc[qi]
    result = proposed_retrieve(
        q["image_path"], proposed_preprocessor, dinov3, hnsw_retriever, gallery_df,
        candidate_k=RERANK_CANDIDATES, final_k=QUAL_TOP_K,
        enable_reranking=ENABLE_RERANKING, alpha_visual=ALPHA_VISUAL, device=DEVICE,
    )
    indices, scores = result["indices"][0], result["scores"][0]

    rows = []
    for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
        if idx < 0:
            continue
        g = gallery_df.iloc[int(idx)]
        rows.append({
            "rank": rank,
            "class_id": int(g["class_id"]),
            "similarity": float(score),
            "correct": int(g["class_id"]) == int(q["class_id"]),
            "image_path": g["image_path"],
        })

    result_df = pd.DataFrame(rows)
    save_path = QUAL_DIR / f"query_{qi:04d}.png"
    show_proposed_retrieval(q, result_df, save_path=save_path)
    plt.close("all")

print("Qualitative directory:", QUAL_DIR)

### Cell 43 — Tổng hợp qualitative accuracy

In [ ]:
qualitative_summary = []

for qi in range(n_qual):
    q = eval_query_df.iloc[qi]
    result = proposed_retrieve(
        q["image_path"], proposed_preprocessor, dinov3, hnsw_retriever, gallery_df,
        candidate_k=RERANK_CANDIDATES, final_k=QUAL_TOP_K,
        enable_reranking=ENABLE_RERANKING, alpha_visual=ALPHA_VISUAL, device=DEVICE,
    )
    indices = result["indices"][0]
    valid = indices >= 0
    retrieved_classes = gallery_df.iloc[indices[valid]]["class_id"].to_numpy()

    qualitative_summary.append({
        "query_index": qi,
        "query_image_id": int(q["image_id"]),
        "query_class_id": int(q["class_id"]),
        "top1_correct": bool(len(retrieved_classes) > 0 and retrieved_classes[0] == q["class_id"]),
        "top5_correct_count": int(np.sum(retrieved_classes == q["class_id"])),
    })

qualitative_summary_df = pd.DataFrame(qualitative_summary)
display(qualitative_summary_df)
qualitative_summary_df.to_csv(OUTPUT_DIR / "qualitative_summary.csv", index=False)
print("Saved:", OUTPUT_DIR / "qualitative_summary.csv")

# FINAL

## 19. Save Experiment Report

`metrics.json`, `latency.json` và `config.json` đã được lưu ở phần
Evaluation (mục 15). Cell dưới chỉ in báo cáo tổng kết ra màn hình.

## 20. Final Summary

### Cell 44 — Final proposed report

In [ ]:
print("=" * 70)
print("PROPOSED EXPERIMENT COMPLETED")
print("=" * 70)

print("\nDataset")
print("Gallery:", len(gallery_df))
print("Query  :", len(eval_query_df))
print("Classes:", gallery_df["class_id"].nunique())

print("\nEncoder")
print("DINOv3 :", MODEL_SOURCE)
print("Feature dimension:", dinov3.dim)

print("\nIndex")
print("FAISS HNSW")
print("M             :", HNSW_M)
print("efConstruction:", HNSW_EF_CONSTRUCTION)
print("efSearch      :", HNSW_EF_SEARCH)

print("\nRe-ranking")
print("Enabled     :", ENABLE_RERANKING)
print("Candidate K :", RERANK_CANDIDATES)
if ENABLE_RERANKING:
    print("alpha_visual:", ALPHA_VISUAL)

print("\nMetrics")
for k in KS:
    key = f"Recall@{k}"
    print(f"Recall@{k}: {metrics[key]:.4f}")
print(f"mAP: {metrics['mAP']:.4f}")

print("\nLatency")
print(f"Encoding    mean: {latency_stats['encoding_mean_ms']:.2f} ms")
print(f"HNSW search mean: {latency_stats['search_mean_ms']:.2f} ms")
print(f"Re-ranking  mean: {latency_stats['rerank_mean_ms']:.2f} ms")
print(f"End-to-end  mean: {latency_stats['end_to_end_mean_ms']:.2f} ms")

print("\nMemory")
print(f"Embedding : {memory_stats['gallery_embeddings_MB']:.2f} MB")
print(f"HNSW index: {memory_stats['hnsw_index_MB']:.2f} MB")

print("\nOutput directory:")
print(OUTPUT_DIR)

# Output expected

Sau khi chạy xong Notebook 03:

```text
outputs/
└── proposed/
    ├── gallery_embeddings.npy
    ├── gallery_metadata.csv
    ├── gallery_hnsw.index
    ├── retrieval_results.npy
    ├── retrieval_scores.npy
    ├── metrics.json
    ├── latency.json
    ├── config.json
    ├── recall_at_k.png
    ├── ablation_hnsw_vs_reranking.csv
    ├── qualitative_query_0000.png
    ├── qualitative_summary.csv
    │
    └── qualitative/
        ├── query_0000.png
        ├── query_0001.png
        └── ...
```

và `data/sampled/` không đổi (Notebook 03 không ghi thêm split mới).

## Bảng tóm tắt

| Section | Input | Output | Offline/Online |
|---|---|---|---|
| DINOv3 extraction | Gallery images | embeddings | Offline |
| HNSW indexing | embeddings | index | Offline |
| Query encoding | Query image | query embedding | Online |
| HNSW search | query embedding | Top-M | Online |
| Re-ranking | Top-M + metadata | Top-K | Online |
| Evaluation | retrieval results | Recall/mAP | Offline |
| Latency | retrieval pipeline | ms | Online |
| Memory | embeddings/index | MB | Offline |

## Liên kết với các notebook khác

- **Notebook 01 — Sampling**: tạo `data/sampled/sop_20k.csv` (20K ảnh SOP,
  stratified theo `class_id`). Notebook 03 chỉ đọc, không sampling lại.
- **Notebook 02 — Baseline**: tạo `data/sampled/baseline_gallery.csv` /
  `baseline_query.csv` — Notebook 03 dùng lại NGUYÊN split này để so sánh
  công bằng, và giữ đúng quy ước đặt tên metric (`Recall@K`, `mAP`) cùng các
  key latency (`encoding_mean_ms`, `search_mean_ms`, `end_to_end_mean_ms`,
  ...) đã dùng ở Notebook 02.
- **Notebook 04 — Comparison/Evaluation**: đọc
  `outputs/baseline/metrics.json` + `latency.json` và
  `outputs/proposed/metrics.json` + `latency.json` để vẽ biểu đồ so sánh
  trực tiếp Recall@K, mAP, latency, memory giữa hai pipeline.
- **Notebook 05 — Qualitative**: có thể tái sử dụng
  `outputs/proposed/qualitative/` và `outputs/baseline/qualitative/` để
  dựng các ví dụ so sánh cạnh nhau (baseline vs proposed) cho cùng một
  query.

In [ ]:
# ==============================================================================
# KAGGLE OUTPUT & RESULTS PREVIEW BLOCK (INDEPENDENT BLOCK)
# ==============================================================================
import os
import shutil
from pathlib import Path

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/working')

if IS_KAGGLE:
    print("--- Kaggle Output & Preview ---")
    
    # 1. Print preview
    print("Preview of proposed metrics:")
    metrics_path = Path("outputs/proposed/metrics.json")
    if metrics_path.exists():
        with open(metrics_path, 'r') as f:
            print(f.read())
            
    # 2. Export to /kaggle/working
    print("\nCopying results to Kaggle output directory (/kaggle/working)...")
    dest = Path("/kaggle/working/outputs/proposed")
    dest.mkdir(parents=True, exist_ok=True)
    src = Path("outputs/proposed")
    if src.exists():
        for item in src.iterdir():
            if item.is_file():
                shutil.copy(item, dest)
                print(f"Copied {item.name} to {dest}")
        qual_src = src / "qualitative"
        qual_dest = dest / "qualitative"
        if qual_src.exists():
            qual_dest.mkdir(parents=True, exist_ok=True)
            for item in qual_src.iterdir():
                if item.is_file():
                    shutil.copy(item, qual_dest)
